# Component classifier on Kaggle

Trains and evaluates the symbol-crop classifier in `src/Object Detection/Component Training/` straight from the GitHub repo. All logic lives in the repo's `.py` files; this notebook only wires Kaggle paths to them.

**Before running**
- Settings → Accelerator: GPU. Settings → Internet: On (for `git` and `pip`).
- Attach the training crops dataset (`manifest.jsonl`, `classes.txt`, `images/train`, `images/val`).
- Optional: attach the CGHD hand-drawn test crops (`images/test`, `labels/test`; `manifest.jsonl` optional). The printed held-out test set ships with the repo, so nothing is downloaded by default.

Run top to bottom. Every cell is safe to re-run after a kernel restart.

In [ ]:
# 1. Repo + data wiring
import os, shutil, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/ishuu19/Single-Line-Diagram-and-Energy-Mapping.git"
REPO = Path("/kaggle/working/Single-Line-Diagram-and-Energy-Mapping")
TRAIN_DIR = REPO / "src" / "Object Detection" / "Component Training"
SAVE_DIR = Path("/kaggle/working/component_train_outputs")   # survives "Save Version"
INPUT = Path("/kaggle/input")

def sh(*cmd, cwd=None):
    r = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    print((r.stdout + r.stderr).strip() or f"$ {' '.join(cmd)}")
    return r.returncode

if (REPO / ".git").is_dir():
    sh("git", "pull", "--ff-only", cwd=REPO)
else:
    sh("git", "clone", "-q", REPO_URL, str(REPO))
assert (TRAIN_DIR / "train.py").exists(), TRAIN_DIR

def find_dir(marker: str):
    """First directory under /kaggle/input containing `marker` (a relative path like images/train)."""
    for hit in INPUT.rglob(marker.split("/")[-1]):
        root = hit
        for _ in marker.split("/"):
            root = root.parent
        if (root / marker).exists():
            return root
    return None

def link(src: Path, dest: Path):
    if dest.is_symlink() or dest.exists():
        dest.unlink() if dest.is_symlink() else shutil.rmtree(dest)
    os.symlink(src, dest, target_is_directory=True)

# training crops: config.py expects REPO/component-symbols (read-only is fine -> symlink)
SYMBOLS = find_dir("images/train")
assert SYMBOLS and (SYMBOLS / "manifest.jsonl").exists(), "training crops dataset not attached"
link(SYMBOLS, REPO / "component-symbols")

# test crops: normalisation rewrites them, and /kaggle/input is read-only -> copy
TEST_DEST = REPO / "component-symbols-test"
TEST_SRC = find_dir("labels/test")
if TEST_SRC and not TEST_DEST.exists():
    shutil.copytree(TEST_SRC, TEST_DEST)

# bring back models from an earlier session of this notebook
if (SAVE_DIR / "models").exists():
    (TRAIN_DIR / "models").mkdir(exist_ok=True)
    for f in (SAVE_DIR / "models").glob("*.keras"):
        if not (TRAIN_DIR / "models" / f.name).exists():
            shutil.copy(f, TRAIN_DIR / "models" / f.name)

os.chdir(TRAIN_DIR)
sys.path.insert(0, str(TRAIN_DIR))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "keras-tuner", "huggingface_hub"], check=True)

import tensorflow as tf
print("train crops:", SYMBOLS)
print("test crops :", TEST_SRC or "not attached (step 4 will download from Hugging Face)")
print("models     :", sorted(p.name for p in (TRAIN_DIR / "models").glob("*.keras")))
print("TF", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

## 2. Data check
`data.py` reads the `train` / `val` splits from `manifest.jsonl`, inverts, pads to 128×128 and one-hot encodes. Expect 64 classes, 13,312 train and 3,328 val crops (260 per type; re-upload the Kaggle dataset after regenerating).

In [ ]:
from config import DEFAULT_HP, DATA_DIR
from data import load_datasets

train_ds, val_ds, class_names, class_weights = load_datasets(mixup_alpha=DEFAULT_HP["mixup_alpha"])
print("DATA_DIR:", DATA_DIR)
print("classes:", len(class_names))
print("train batches:", int(train_ds.cardinality()), "| val batches:", int(val_ds.cardinality()))
print("class weights all 1.0:", all(abs(w - 1.0) < 1e-6 for w in class_weights.values()))

## 3. Train
Warmup + cosine decay over `EPOCHS`, per-sample `RandomDegrade`, MixUp, label smoothing, AdamW. Writes `models/custom.keras`, `models/custom_best.keras`, `outputs/custom_history.csv`, `logs/custom/`. ~80 epochs is about 25 min on a P100. Set `EPOCHS = 3` for a smoke test.

Skip this cell to evaluate the committed checkpoint `models/notebook_best.keras` instead.

In [ ]:
EPOCHS = 80
!python train.py --epochs {EPOCHS}

## 4. Evaluate: val, printed held-out test, then CGHD hand-drawn test
`evaluate.py` reports clean / TTA / degraded accuracy plus a classification report and confusion matrix for each split.

- `val`: Schematex crops in the training style.
- `printed-test`: 192 crops (3 per type, 64 types) in `component-symbols-test-printed/` (in the repo) rendered with fonts, stroke widths, paper tints, scales, a 180° orientation and zoom the model never saw. Printed, not hand drawn. This is the first real generalisation number.
- `cghd-test`: only if the CGHD dataset is attached, or `FETCH_CGHD_SUBSET = True` (downloads 20 random sheets out of the first 100, not the whole dataset). Scored raw, then after `normalise_test_crops.py`.

In [ ]:
import json, os, random
from pathlib import Path
from IPython.display import Image as IPyImage, display

FETCH_CGHD_SUBSET = False   # True: fetch 20 random CGHD sheets from the first 100 on the Hub (needs Internet)

have = {p.stem for p in Path("models").glob("*.keras")}
MODEL_TAG = next(t for t in ("custom_best", "custom", "notebook_best") if t in have)
print("model:", MODEL_TAG, "| available:", sorted(have))

!python evaluate.py --model {MODEL_TAG} --split val
!python evaluate.py --model {MODEL_TAG} --split printed-test

if not (TEST_DEST / "images" / "test").is_dir() and FETCH_CGHD_SUBSET:
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
    from huggingface_hub import list_repo_files, hf_hub_download
    CGHD_LOCAL = Path("/kaggle/working/cghd_subset")
    files = list_repo_files("lowercaseonly/cghd", repo_type="dataset")
    images = sorted(f for f in files if "/images/" in f and f.lower().endswith((".jpg", ".jpeg", ".png")))[:100]
    for img in random.Random(0).sample(images, 20):
        ann = img.replace("/images/", "/annotations/").rsplit(".", 1)[0] + ".xml"
        for rel in (img, ann):
            hf_hub_download("lowercaseonly/cghd", rel, repo_type="dataset", local_dir=CGHD_LOCAL)
    !python build_cghd_test_crops.py --cghd-root {CGHD_LOCAL} --pad 0.12 --refresh

if (TEST_DEST / "images" / "test").is_dir():
    if not (TEST_DEST / "images" / "test_raw").is_dir():
        !python evaluate.py --model {MODEL_TAG} --split cghd-test --out-suffix _raw
    !python normalise_test_crops.py --preview 6
    !python evaluate.py --model {MODEL_TAG} --split cghd-test
    display(IPyImage("outputs/test_crops_normalised.png"))
else:
    print("CGHD crops not attached; skipped")

for tag in ("val", "printed_test", "cghd_test_raw", "cghd_test"):
    p = Path("outputs") / f"{MODEL_TAG}_{tag}_metrics.json"
    if p.exists():
        print(f"{tag:<14}", json.loads(p.read_text()))

## 5. Look at the printed test predictions
Accuracy broken down by the style factor that changed (orientation, scale, font, paper), then a grid of examples (green = correct). Set `TEST_DIR = config.TEST_DATA_DIR` to inspect the CGHD crops instead.

In [ ]:
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

import model as _  # registers RandomDegrade
import data, config

TEST_DIR = config.PRINTED_TEST_DATA_DIR
_, _, class_names, _ = data.load_datasets()
test_ds = data.load_external_test_dataset(class_names, TEST_DIR)
net = tf.keras.models.load_model(str(config.MODEL_DIR / f"{MODEL_TAG}.keras"), compile=False)

images, y_true, y_pred, conf = [], [], [], []
for x, y in test_ds:
    p = tf.nn.softmax(net(x, training=False), -1).numpy()
    images.append(x.numpy()); y_true.append(np.argmax(y.numpy(), -1))
    y_pred.append(np.argmax(p, -1)); conf.append(p.max(-1))
images, y_true, y_pred, conf = map(np.concatenate, (images, y_true, y_pred, conf))

ok = y_true == y_pred
print(f"accuracy: {ok.mean():.3f} on {len(y_true)} crops")
rows = [r for r in data.read_test_manifest(TEST_DIR) if r["type"] in class_names]
for key in ("orientation", "scale", "font", "paper", "standard"):
    if rows and key in rows[0]:
        by = {}
        for r, o in zip(rows, ok):
            by.setdefault(r[key], []).append(o)
        print(f"{key:<12}", {k: f"{np.mean(v):.2f} (n={len(v)})" for k, v in sorted(by.items(), key=str)})

n, cols = min(24, len(images)), 6
idx = np.random.RandomState(0).choice(len(images), n, replace=False)
rows = int(np.ceil(n / cols))
fig, axes = plt.subplots(rows, cols, figsize=(2.2 * cols, 2.6 * rows), squeeze=False)
for ax, i in zip(axes.flat, idx):
    ax.imshow(255 - images[i].squeeze(), cmap="gray")
    ok = y_true[i] == y_pred[i]
    ax.set_title(f"{class_names[y_true[i]]}\n-> {class_names[y_pred[i]]} ({conf[i]:.2f})",
                 fontsize=8, color="green" if ok else "red")
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout(); plt.show()

## 6. Keep the artefacts
Copies `models/`, `outputs/`, `logs/` to `/kaggle/working/component_train_outputs`, which Kaggle stores with the notebook version. Download `outputs/*_metrics.json` and `models/custom_best.keras` from there and commit them to the repo by hand.

In [ ]:
for name in ("models", "outputs", "logs"):
    src = TRAIN_DIR / name
    if src.exists():
        shutil.copytree(src, SAVE_DIR / name, dirs_exist_ok=True)
print(sorted(str(p.relative_to(SAVE_DIR)) for p in SAVE_DIR.rglob("*") if p.is_file()))